# Week 90, Day 5: Code review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w090d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('sharing_draft.py', 'w', encoding='utf-8') as f:
    f.write('import hashlib\nimport logging\nimport pickle\nimport random\nimport sqlite3\n\nlog = logging.getLogger("waymark.sharing")\n\n\ndef new_share_code():\n    return "".join(random.choice("abcdefghijklmnopqrstuvwxyz234567") for _ in range(16))\n\n\ndef check_code(stored_digest, code):\n    digest = hashlib.sha256(code.encode()).hexdigest()\n    return digest == stored_digest\n\n\ndef open_share(conn, code):\n    log.info("opening share %s", code)\n    row = conn.execute(f"SELECT route FROM share WHERE digest = \'{hashlib.sha256(code.encode()).hexdigest()}\'").fetchone()\n    return row[0] if row else None\n\n\ndef import_settings(blob):\n    return pickle.loads(blob)\n\n\ndef make_reset_token(user):\n    token = str(random.randint(100000, 999999))\n    print("reset token for", user, token)\n    return token\n')

**Predict before you run:** what will this print?



In [ ]:
import re

lines = ["GET /s/mfrggzdfmztwq2lk 200", "POST /login password=hunter2 302", "GET /v1/walks?token=abc123&limit=5 200"]
for line in lines:
    print(re.sub(r"(password|token)=[^&\s]+", r"\1=***", line))

> **Key idea.** Redact secrets at the source before logging (paths like `/s/CODE`, `token=`, `password=`, secret JSON fields), log identifiers instead of secrets, and never print them. Check ownership on every record access in the service or repository, and answer others' records with 404. Automate the recognisable shapes: `random` for secrets, `==` on secrets, string-built SQL, `pickle.loads` and `eval` on input, secrets passed to print or loggers. Pin dependencies and audit them against advisories in CI. Tools find candidates; people judge them.

### Exercise: Redacting logs

Write `redact(line)`: replace share codes in paths (`/s/` followed by letters and digits) with `/s/***`; the values of query or form parameters named `token`, `password`, `code` or `key` (up to the next `&`, space or end) with `***`; and the values of JSON string fields with those names (`"token": "..."`) with `"***"`. Leave everything else alone.

In [ ]:
import re

def redact(line):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_redact():\n    \"no secrets in logs\"\n    assert redact(\"GET /s/mfrggzdfmztwq2lk 200 12ms\") == \"GET /s/*** 200 12ms\"\n    assert redact(\"POST /login password=hunter2&user=ada 302\") == \"POST /login password=***&user=ada 302\"\n    assert redact(\"GET /v1/walks?token=abc123&limit=5 200\") == \"GET /v1/walks?token=***&limit=5 200\"\n    assert redact('body {\"code\": \"4f2a9\", \"km\": 6.0, \"key\": \"k-1\"}') == 'body {\"code\": \"***\", \"km\": 6.0, \"key\": \"***\"}'\n    assert redact(\"GET /v1/walks?limit=5&monkey=3 200\") == \"GET /v1/walks?limit=5&monkey=3 200\"")

### Exercise: Only your own walks

Write `WalkRepository` (in memory) with `add(owner, walk)` returning the new id (from 1), `get(owner, walk_id)` returning the walk only if it belongs to `owner` (else `None`, the same for missing and someone else's), `list(owner)` returning that owner's walks in id order, and `delete(owner, walk_id)` returning whether one of the owner's walks was deleted. Stored walks include their `id` but not their owner.

In [ ]:
class WalkRepository:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ownership():\n    \"nobody else's walks\"\n    repo = WalkRepository()\n    a = repo.add(\"ada\", {\"km\": 6.0})\n    b = repo.add(\"bo\", {\"km\": 9.0})\n    c = repo.add(\"ada\", {\"km\": 3.0})\n    assert (a, b, c) == (1, 2, 3)\n    assert repo.get(\"ada\", 1) == {\"id\": 1, \"km\": 6.0} and repo.get(\"ada\", 2) is None and repo.get(\"ada\", 99) is None\n    assert [w[\"id\"] for w in repo.list(\"ada\")] == [1, 3] and repo.list(\"cy\") == []\n    assert repo.delete(\"ada\", 2) is False and repo.get(\"bo\", 2) == {\"id\": 2, \"km\": 9.0}\n    assert repo.delete(\"bo\", 2) is True and repo.get(\"bo\", 2) is None\n    assert \"owner\" not in repo.get(\"ada\", 1)")

### Exercise: Finding insecure shapes

Write `insecure_random(source)`: a list of `(function, call)` for calls to `random.random`, `random.choice`, `random.randint`, `random.randrange`, `random.choices` or `random.Random` inside top-level functions whose names contain `token`, `code`, `key`, `secret` or `password`, in source order (`call` written as `"random.choice"`). Then `plain_secret_compares(source)`: a list of `(function, line)` for `==` or `!=` comparisons in any top-level function where either side is a name containing `token`, `digest`, `signature`, `secret` or `code`.

In [ ]:
import ast

def insecure_random(source):
    ___

def plain_secret_compares(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_shapes():\n    \"in this sprint's draft\"\n    src = open(\"sharing_draft.py\").read()\n    assert insecure_random(src) == [(\"new_share_code\", \"random.choice\"), (\"make_reset_token\", \"random.randint\")]\n    assert plain_secret_compares(src) == [(\"check_code\", 16)]\n    safe = \"import secrets, hmac\\n\\ndef new_code():\\n    return secrets.token_urlsafe(10)\\n\\ndef same(a, digest):\\n    return hmac.compare_digest(a, digest)\\n\\ndef shuffle_walks(walks):\\n    import random\\n    return random.choice(walks)\\n\"\n    assert insecure_random(safe) == [] and plain_secret_compares(safe) == []")

### Exercise: Auditing dependencies

Write `audit(lock_text, advisories)`: `lock_text` has one pinned requirement per line (`name==version`; ignore blank lines and lines starting with `#`; names compare in lower case); `advisories` maps a package name to a list of `(advisory_id, fixed_in)` meaning versions **below** `fixed_in` are affected. Return a sorted list of `"NAME VERSION: ID (fixed in FIXED)"`, comparing versions as tuples of integers.

In [ ]:
def audit(lock_text, advisories):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("LOCK = \"\"\"# waymark requirements, pinned\nrequests==2.30.0\nRich==13.7.1\n\nurllib3==2.2.1\npyyaml==5.3\n\"\"\"\nADVISORIES = {\"requests\": [(\"PYSEC-2023-74\", \"2.31.0\")], \"urllib3\": [(\"PYSEC-2024-1\", \"2.2.2\"), (\"PYSEC-2023-9\", \"2.0.6\")],\n              \"pyyaml\": [(\"PYSEC-2021-142\", \"5.4\")], \"rich\": []}\n\ndef test_audit():\n    \"known holes in pinned versions\"\n    assert audit(LOCK, ADVISORIES) == [\"pyyaml 5.3: PYSEC-2021-142 (fixed in 5.4)\", \"requests 2.30.0: PYSEC-2023-74 (fixed in 2.31.0)\",\n                                       \"urllib3 2.2.1: PYSEC-2024-1 (fixed in 2.2.2)\"]\n    assert audit(\"requests==2.31.0\\n\", ADVISORIES) == []")

### Bug hunt: A share code in the log

The access log writes the whole path, including the share code, so anyone who can read the logs can open the shared route: it prints `GET /s/mfrggzdfmztwq2lk 200`. Redact the code before logging, so it prints `GET /s/*** 200`.

In [ ]:
import re

def log_line(method, path, status):
    return f"{method} {path} {status}"

print(log_line("GET", "/s/mfrggzdfmztwq2lk", 200))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"redact at the source\"\n    expect_output(\"GET /s/*** 200\")")

### Bug hunt: Anyone's walk by number

The handler returns any walk by its id, whoever asks, so Bo can read Ada's walk: it prints `bo sees: {'km': 6.0}`. Check the owner, and answer with `None` (a 404) for other people's walks, so it prints `bo sees: None`.

In [ ]:
WALKS = {17: {"owner": "ada", "km": 6.0}}

def get_walk(user, walk_id):
    walk = WALKS.get(walk_id)
    return {"km": walk["km"]} if walk else None

print("bo sees:", get_walk("bo", 17))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"ownership on every access\"\n    expect_output(\"bo sees: None\")")

### Puzzle: Opening a share, fixed

Arrange the fixed `open_share`: a parameterised query on the code's hash, no secret in the log, and no plain comparison. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the acronym for reading other people's records by changing an id (four capital letters)
q = ___   # the module to use instead of random for share codes
r = ___   # the function to use instead of == for comparing digests (module.function)
s = ___   # True or False: pickle.loads is safe on data from a user's upload
t = ___   # the status for another user's walk: 403 or 404

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"IDOR\" and q == \"secrets\" and r == \"hmac.compare_digest\", \"p, q and r\"\n    assert s is False and t == 404, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a security reviewer

Write `security_review(source)` using the given helpers (and your own `ast` walking). Return `(findings, verdict)`: `findings` is a list of `"LINE SEVERITY: MESSAGE (FUNCTION)"` strings sorted by line, then message, with these rules: `high: random used for a secret` for each `insecure_random` call (at the call's line); `high: secret compared with ==` for each `plain_secret_compares` result; `high: SQL built from a string` for any `execute` call whose first argument is an f-string (`ast.JoinedStr`) or a `%` formatting (`ast.BinOp` with `%`); `high: pickle.loads on outside data` for any call to `pickle.loads`; `medium: secret written to output` for any call to `print` or to a method `debug`, `info`, `warning` or `error` of a name, with an argument that's a name containing `token`, `code`, `password` or `secret`. `verdict` is `"block"` if any finding is high, `"fix soon"` if any is medium, else `"ok"`.

In [ ]:
import ast

SECRET_FUNCS = ("token", "code", "key", "secret", "password")
SECRET_NAMES = ("token", "digest", "signature", "secret", "code")
RANDOM_CALLS = {"random", "choice", "randint", "randrange", "choices", "Random"}

def _functions(tree):
    return [n for n in tree.body if isinstance(n, ast.FunctionDef)]

def insecure_random(tree):
    found = []
    for func in _functions(tree):
        if any(word in func.name for word in SECRET_FUNCS):
            for node in ast.walk(func):
                if (isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and isinstance(node.func.value, ast.Name)
                        and node.func.value.id == "random" and node.func.attr in RANDOM_CALLS):
                    found.append((node.lineno, func.name))
    return found

def plain_secret_compares(tree):
    secretish = lambda n: isinstance(n, ast.Name) and any(w in n.id for w in SECRET_NAMES)
    found = []
    for func in _functions(tree):
        for node in ast.walk(func):
            if isinstance(node, ast.Compare) and any(isinstance(op, (ast.Eq, ast.NotEq)) for op in node.ops):
                if secretish(node.left) or any(secretish(c) for c in node.comparators):
                    found.append((node.lineno, func.name))
    return found

def security_review(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_draft():\n    \"the sprint's draft module\"\n    findings, verdict = security_review(open(\"sharing_draft.py\").read())\n    assert findings == [\n        \"11 high: random used for a secret (new_share_code)\",\n        \"16 high: secret compared with == (check_code)\",\n        \"20 medium: secret written to output (open_share)\",\n        \"21 high: SQL built from a string (open_share)\",\n        \"26 high: pickle.loads on outside data (import_settings)\",\n        \"30 high: random used for a secret (make_reset_token)\",\n        \"31 medium: secret written to output (make_reset_token)\",\n    ]\n    assert verdict == \"block\"\n\ndef test_verdicts():\n    \"fix soon, and ok\"\n    medium = \"import logging\\nlog = logging.getLogger()\\n\\ndef send(token):\\n    log.info('sent %s', token)\\n\"\n    assert security_review(medium) == ([\"5 medium: secret written to output (send)\"], \"fix soon\")\n    clean = \"import secrets\\n\\ndef new_code():\\n    return secrets.token_urlsafe(10)\\n\\ndef count(conn):\\n    return conn.execute('SELECT COUNT(*) FROM walk').fetchone()\\n\"\n    assert security_review(clean) == ([], \"ok\")")

## Warm-up

The share code is still in the first line, in plain sight in a log file that many people and tools can read. Anything that works as a key (share codes, tokens, passwords, reset codes) must never reach a log, an error message or an analytics event. Open `sharing_draft.py` above: it's a pull request from this sprint. Count its problems before reading on.

## Secrets in logs

Logs are copied, shipped to other services, kept for months and read by people who'd never be given the database. So **redact** at the source: a function that replaces secret values (share codes in `/s/...` paths, `token=`, `password=`, `code=` parameters, and JSON fields with those names) with `***` before anything is written. Better still, log **identifiers** that aren't secrets (a share's database id, or the first few characters of the code's hash) rather than the secret itself. And never `print` a secret "just while debugging": it ends up in a log too.

## Whose record is it?

**Insecure direct object reference** (IDOR) is the most common access-control bug: `GET /v1/walks/17` returns walk 17 to **anyone** signed in, so changing the number reads other people's data. Every access to a user's record checks ownership, in the service or repository (`WHERE id = ? AND owner = ?`), not just in the interface. Answer another user's record with the same `404` as a missing one, so ids can't be probed.

## What a machine can find

Some security bugs have a recognisable shape, which the syntax tree can find:

- **Insecure randomness**: `random.random`, `random.choice`, `random.randint` and `random.Random` in a function that makes a token, code, key or password (use `secrets`).
- **Plain comparisons on secrets**: `==` or `!=` where one side's name mentions a token, digest, signature or code (use `hmac.compare_digest`).
- **SQL built from strings**: f-strings or `%` formatting passed to `execute`.
- **Dangerous calls**: `pickle.loads` on data from outside (it can run any code), `eval` and `exec` on input, `yaml.load` without a safe loader, `subprocess` with `shell=True`.
- **Secrets printed or logged**: a call to `print` or a logger method with an argument named like a secret.

A tool such as **Bandit** does this (and much more) on your own computer; today's version shows how it works, and like Week 88's reviewer it finds candidates for a person to judge.

## Dependencies

Code you didn't write runs with the same rights as yours. Pin your dependencies (Week 73), and **audit** them against published advisories: each advisory names a package, the affected versions and the version that fixes it. `pip-audit` does this against the Python advisory database on your own computer; in CI it can fail a build that depends on a vulnerable version.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **A security review** of your capstone with today's list: logs, ownership, randomness, comparisons, string-built queries, dangerous calls.
2. **Run Bandit and pip-audit** on your own computer (both free), and add them to CI.
3. **Fix every high finding** before the sprint review, each in its own small pull request with a test.

## Before you go

Tomorrow is the last sprint review: the feature freeze. After it, nothing new is built: the next week hardens what exists.